# LoL Scrape

Pulls gol.gg team, player, champion, and game data for LCK/LPL/LEC/LCS
and caches it into dated data_cache snapshots. Run this first; every
other notebook reads its cached output tables. See
docs/superpowers/specs/2026-09-06-lol-scrape-design.md for the design
this notebook implements.

Set TEST_MODE = True for a first run: caps to one region and its first
2 teams so the whole path (list tables, team matchlist, game manifest,
game parse, hash guard, snapshot write) can be checked cheaply before a
full run.

Caveat: don't flip TEST_MODE back on for a region already fully scraped
today. The teams.csv hash guard compares content, so a smaller-scope
teams_df (fewer teams) legitimately hashes differently from a fuller one
and will still overwrite today's snapshot with the smaller version.
data_cache/ is gitignored and fully regenerable, so this is a caveat to
avoid, not a guard worth adding extra state-tracking logic for.

In [1]:
from __future__ import annotations

import csv
from pathlib import Path

import pandas as pd
import requests

import lol_lib
import lol_scrape_lib

## 2. Parameters

In [2]:
SEASON = "S16"
SPLIT = "Summer"
TEST_MODE = True
TEST_MODE_TEAM_LIMIT = 2

# Real gol.gg tournament name per region (not the league abbreviation) -
# fetching by tournament name is what actually narrows to that league's
# real roster; gol.gg's Region column is a server region and also
# contains that server's academy/challenger/collegiate teams. Update
# this by hand for a different competitive period; leagues do not share
# one split-naming convention (LPL uses numbered splits, LCK currently
# uses round-based stages, not Spring/Summer), so there is no single
# SPLIT value that works for all four.
TOURNAMENTS = {
    "LCK": "LCK 2026 Rounds 3-4",
    "LEC": "LEC 2026 Summer Season",
    "LCS": "LCS 2026 Summer",
    "LPL": "LPL 2026 Split 3",
}

REGIONS = lol_lib.REGIONS if not TEST_MODE else lol_lib.REGIONS[:1]

## 3. Session setup

In [3]:
lol_lib.print_versions()
lol_lib.setup_output_dirs()
session = requests.Session()

Package versions:
python: 3.14.5
pandas: 2.3.3
numpy: 2.4.6
matplotlib: 3.11.0
requests: 2.34.2


## 4. Players and champions (per-region, concatenated)

Not truly "global" despite living in `data_cache/global/`: each region is
fetched separately by its real tournament name (same mechanism as the
per-region teams fetch below) and concatenated, since Task 9 found that a
season-wide, unfiltered fetch mixed every league on earth together and did
not even share one split-naming convention with `games.csv`. Kept under
`global_scrape_snapshot_dir()` rather than a per-region folder since the
saved file is still one combined table, not four separate ones.

In [4]:
players_df = pd.concat(
    [lol_scrape_lib.scrape_global_list("players", region, SEASON, SPLIT, TOURNAMENTS[region], session) for region in REGIONS],
    ignore_index=True,
)
champions_df = pd.concat(
    [lol_scrape_lib.scrape_global_list("champion", region, SEASON, SPLIT, TOURNAMENTS[region], session) for region in REGIONS],
    ignore_index=True,
)

global_tables = {"players": players_df, "champions": champions_df}
latest_global = lol_lib.latest_global_scrape_snapshot()

# Both files must exist for the comparison to mean anything: an empty
# generator (from the old "if exists()" filter alone) makes all() vacuously
# True, so an interrupted run that left an empty or half-written folder
# would make every future run silently skip writing forever.
if latest_global is not None and all((latest_global / f"{name}.csv").exists() for name in global_tables) and all(
    lol_scrape_lib.hash_table(df) == lol_scrape_lib.hash_table(
        # keep_default_na=False, matching the convention parse_list_table
        # already uses: without it, a literal "NA" (e.g. the LCS region
        # code elsewhere in this pipeline) reads back as a missing value,
        # which drifts the reloaded table's dtypes from the freshly
        # scraped one and breaks this hash comparison.
        pd.read_csv(latest_global / f"{name}.csv", keep_default_na=False, dtype=str)
    )
    for name, df in global_tables.items()
):
    print(f"no changes to global lists since {latest_global.name}, skipping snapshot")
else:
    snapshot_dir = lol_lib.global_scrape_snapshot_dir()
    snapshot_dir.mkdir(parents=True, exist_ok=True)
    for name, df in global_tables.items():
        df.to_csv(snapshot_dir / f"{name}.csv", index=False)
    print(f"wrote global snapshot to {snapshot_dir.relative_to(lol_lib.PROJECT_ROOT)}")

wrote global snapshot to data_cache\global\2026-09-07


## 5. Per-region teams and games

In [5]:
# Sections 5 and 6 (teams fetch, hash guard, resumable game fetch) all live
# in this one cell: a `for region in REGIONS:` loop's body can't span
# separate notebook cells (each cell is its own independently-executed
# unit), so splitting it would only run the later phases once total, using
# whatever the loop left in scope, instead of once per region.
for region in REGIONS:
    print(f"--- {region} ---")
    tournament = TOURNAMENTS[region]

    teams_df = lol_scrape_lib.scrape_region_teams(region, SEASON, SPLIT, tournament, session)
    if TEST_MODE:
        teams_df = teams_df.head(TEST_MODE_TEAM_LIMIT)

    # Snapshots are self-consistent full sets (spec DATA FLOW step 4): every
    # dated folder always gets a full teams.csv and games.csv, even when
    # nothing changed, rather than a folder missing one of the two tables.
    # games.csv carries forward across days: today's file starts as a copy
    # of the latest snapshot's rows (so already-known games are never
    # re-fetched), written immediately so a crash during today's run still
    # loses at most the one new game in flight, matching spec step 3.
    latest_region = lol_lib.latest_scrape_snapshot(region)
    snapshot_dir = lol_lib.scrape_snapshot_dir(region)
    snapshot_dir.mkdir(parents=True, exist_ok=True)
    games_path = snapshot_dir / "games.csv"
    failures_path = snapshot_dir / "failures.csv"

    # teams_prior_hash is captured before teams_df is written below, so a
    # same-day rerun compares against the PRIOR run's content, not against
    # the file this run is about to write to itself (which would always
    # read back as "unchanged" and make the printed status uninformative).
    teams_prior_hash = None
    if latest_region is not None and (latest_region / "teams.csv").exists():
        # keep_default_na=False/dtype=str: pd.read_html and pd.read_csv
        # infer dtypes differently for identical source text (and pandas
        # treats the literal string "NA", the LCS region code, as missing
        # data by default), so this is the only way two genuinely
        # identical tables hash the same via hash_table.
        teams_prior_hash = lol_scrape_lib.hash_table(pd.read_csv(latest_region / "teams.csv", keep_default_na=False, dtype=str))

    known_rows: list[dict] = []
    if latest_region is not None and (latest_region / "games.csv").exists():
        known_rows = pd.read_csv(latest_region / "games.csv", keep_default_na=False, dtype=str).to_dict("records")

    # A stale prior-schema games.csv carried forward as known_rows would
    # write today's header from the OLD schema while append_row later
    # writes rows in whatever schema assemble_game_row currently returns,
    # producing a corrupted file with no exception raised. Fail loud here
    # instead, matching this project's _require_columns precedent.
    if known_rows and set(known_rows[0].keys()) != set(lol_scrape_lib.GAMES_ROW_COLUMNS):
        raise ValueError(
            f"{region}: latest snapshot's games.csv schema does not match the current "
            "code (a column was added/removed/renamed since that snapshot). Move or "
            "delete the stale data_cache folder before rerunning, since carrying its "
            "rows forward would silently corrupt today's file."
        )

    # Only a permanent parse failure carries forward as "already handled":
    # a transient fetch failure (network timeout, 5xx after retries) must
    # NOT be, since it might succeed on a later run. fetch_html's error
    # always starts with "Failed to fetch" for transient failures;
    # anything else comes from a parse_game_* function, a real gol.gg data
    # defect that will not resolve on its own (confirmed live: game 80200's
    # failure is an actually-empty champion in gol.gg's own markup, not a
    # parser bug). A transient failure that's still failing gets
    # rediscovered and freshly logged by this run's own attempt instead.
    known_failures: list[dict] = []
    if latest_region is not None and (latest_region / "failures.csv").exists():
        all_known_failures = pd.read_csv(latest_region / "failures.csv", keep_default_na=False, dtype=str).to_dict("records")
        known_failures = [row for row in all_known_failures if not row["error"].startswith("Failed to fetch")]

    already_fetched_ids = {row["game_id"] for row in known_rows} | {row["game_id"] for row in known_failures}
    # GAMES_ROW_COLUMNS (not known_rows[0].keys()) is the header at both
    # write sites below, so a pure column reorder in a carried-forward file
    # can't slip past the schema-equality guard above and then corrupt a
    # later row written under a mismatched header.
    if known_rows:
        with open(games_path, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=lol_scrape_lib.GAMES_ROW_COLUMNS)
            writer.writeheader()
            writer.writerows(known_rows)
    if known_failures:
        with open(failures_path, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=["game_id", "url", "error"])
            writer.writeheader()
            writer.writerows(known_failures)

    def append_row(row: dict, path: Path = games_path) -> None:
        is_new_file = not path.exists()
        with open(path, "a", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=lol_scrape_lib.GAMES_ROW_COLUMNS)
            if is_new_file:
                writer.writeheader()
            writer.writerow(row)

    def append_failure(failure: dict, path: Path = failures_path) -> None:
        is_new_file = not path.exists()
        with open(path, "a", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=["game_id", "url", "error"])
            if is_new_file:
                writer.writeheader()
            writer.writerow(failure)

    lol_scrape_lib.scrape_region_games(
        region, SEASON, SPLIT, tournament, teams_df, session,
        already_fetched_ids, on_row=append_row, on_failure=append_failure,
    )

    teams_df.to_csv(snapshot_dir / "teams.csv", index=False)

    teams_changed = teams_prior_hash is None or lol_scrape_lib.hash_table(teams_df) != teams_prior_hash
    # games_path may not exist yet if this region had no prior snapshot and
    # discovered zero games this run (e.g. a not-yet-started tournament).
    games_added = games_path.exists() and len(known_rows) < len(pd.read_csv(games_path, keep_default_na=False, dtype=str))
    relative_snapshot_dir = snapshot_dir.relative_to(lol_lib.PROJECT_ROOT)
    if teams_changed or games_added:
        print(f"finished {region}: snapshot at {relative_snapshot_dir} (changed)")
    else:
        print(f"finished {region}: snapshot at {relative_snapshot_dir} (no changes since {latest_region.name})")

--- LCK ---


finished LCK: snapshot at data_cache\LCK\2026-09-07 (changed)
